🟢 PHASE 1 — DataFrame Basics (Must Master)

Equivalent of Pandas basics.

In [0]:
df = spark.read.csv('/Volumes/pyspark_pracitce/default/pyspark/train.csv',header = True, inferSchema = True)
display(df)

In [0]:
#Viewing data
df.show()

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import expr #type casting changing datatype

df = df.withColumn("Sales", expr("try_cast(Sales as float)"))

In [0]:
df.printSchema() #printing the structural tree of the dataframe

In [0]:
df.describe()

##Selecting columns

In [0]:
df.select('Sales').show()

In [0]:
from pyspark.sql.functions import col

df_selected_col = df.select(col("Row ID"), col("Customer Name"),col("Sales"))
df_selected_col.show()

##Filtering

In [0]:
df_filter_row = df.filter(df['Row ID'] == 1)
df_filter_row.show()

In [0]:
df_filtered = df.filter((col("Category") == "Technology") & (col("Sales").cast("double")>50)).orderBy(col("Sales").cast("double").desc())
display(df_filtered)

In [0]:
df_clean = df.where(col("Sales").isNotNull())
print(f"Total rows : {df.count()}")
print(f"Clean rows : {df_clean.count()}")

##Adding and Modifing

In [0]:
from pyspark.sql.functions import lit  # for adding a new column 
df_withstatus = df_clean.withColumn("Status", lit("Active"))
df_withstatus.select("Customer Name", "Status").show(5)

In [0]:
df_double = df_clean.withColumn("Status",col("Sales")*2)
df_double.select("Customer Name", "Status").show(5)

In [0]:
from pyspark.sql.functions import when
# condition wise creating column
df_category = df_double.withColumn("Sales_category",
                                   when(col("Sales")>500,"High")
                                   .when(col("Sales")<100,"Low")
                                   .otherwise("Medium"))
df_category.select("Customer Name", "Sales", "Sales_category").show(10)

In [0]:
display(df_category)

##Droping columns

In [0]:
df_droped = df.drop("Row ID")
df_droped.printSchema()

In [0]:
display(df_droped)

In [0]:
df_renamed = df.withColumnRenamed("Row ID","Row NO")
df_renamed.printSchema()
display(df_renamed)

##PHASE 2: Data Cleaning. This is where you transform "raw" data into "usable" data.

In [0]:
from pyspark.sql.functions import count,when,isnull

In [0]:
#finding null values
df_null = df.select([count(when(isnull(c), c)).alias(c) for c in df.columns])
display(df_null)

In [0]:
df_no_nulls = df.na.drop(subset=['Postal Code'])
display(df_no_nulls.select([count(when(isnull(c), c)).alias(c) for c in df.columns]))

In [0]:
df_filled = df.na.fill({"Postal Code":0, "Sales":0.0})
display(df_filled.select([count(when(isnull(c), c)).alias(c) for c in df.columns]))

In [0]:
df_filled = df.na.fill({"Postal Code":0, "Sales":0.0})
display(df_filled.select([count(when(isnull(c), c)).alias(c) for c in df.columns]))

###Duplicate Handling


In [0]:
df_unique = df.dropDuplicates()
print("before droping duplicate {}".format(df.count()))
print("after droping duplicate {}".format(df_unique.count()))

In [0]:
display(df_unique)

In [0]:
df_unique.printSchema()

In [0]:
df.withColumn('Sales',df['Sales'].cast('int')).printSchema()

In [0]:
from pyspark.sql.functions import trim,lower,col,regexp_replace
df_str_clean = df_unique.withColumn('Customer Name',trim(col('Customer Name'))).withColumn('Category',lower(col('Category'))).withColumn("Product Name",regexp_replace(col("Product Name"), " ", "-"))
df_str_clean.select('Customer Name','Category','Product Name').show()

In [0]:
display(df_unique)

Date Transformation

In [0]:
from pyspark.sql.functions import year, month, dayofmonth, hour, weekofyear, date_format,datediff
df_dates = df_unique.withColumn("Order_year",year(col("Order Date"))).withColumn("Order_month",month(col("Order Date"))).withColumn("Order_day",dayofmonth(col("Order Date"))).withColumn("Order_hour",hour(col("Order Date"))).withColumn("Order_week",weekofyear(col("Order Date"))).withColumn("Days_to_Ship",datediff(col("Ship Date"),col("Order Date")))

In [0]:
display(df_dates)

🟠 PHASE 3 — Aggregations & Business Logic

VERY IMPORTANT for Data Engineer jobs.

In [0]:
from pyspark.sql import functions as F # group by function
category_stats = df_unique.groupBy('Category').agg(F.sum('Sales').alias('Total_Sales'),F.avg('Sales').alias('Avg_Sales'),F.count("Order ID").alias("Order_count"))
category_stats.show()

In [0]:
high_value_stats = df_unique.groupBy("Region").agg(F.sum("Sales").alias("Total_Region_Sales"),F.sum(F.when(F.col("Sales")>500,F.col("Sales")).otherwise(0)).alias("Total_High_Value_Sales"))

high_value_stats.show()
                                                                                                                    
                                                                                                                                                                  
                                                                                                        

In [0]:
from pyspark.sql.window import Window
# 1. Define the "Window" (Group by Category, Order by Sales)
windowSpec = Window.partitionBy("Category").orderBy(F.col("Sales").desc())
# 2. Apply rank() or row_number()
df_ranked = df_unique.withColumn("rank", F.row_number().over(windowSpec))
# 3. Business Question: Show the Top 3 best-selling products in each Category
df_ranked.filter(col("rank") <= 3).select("Category","Product Name","Sales","rank").show()

In [0]:
df_ranked.select(col('Category'),col('Sales'),col('rank')).show()

In [0]:
df_all_rank = df_unique.withColumn("rank", F.row_number().over(windowSpec)).withColumn("rank2",F.rank().over(windowSpec)).withColumn("dense_rank",F.dense_rank().over(windowSpec))

df_all_rank.select(col('Category'),col('Sales'),col('rank'),col('rank2'),col('dense_rank')).show()


In [0]:
# window for a specific cutomer, ordered by time
custwindow = Window.partitionBy("Customer ID").orderBy(col("Order Date"))

df_time = df_unique.withColumn("prev_order_date",F.lag(col("Order Date"),2).over(custwindow)) \
.withColumn("next_order_date",F.lead(col("Order Date"),3).over(custwindow)) \
.withColumn("Days_between_orders",F.datediff(col("Order Date"),F.col("prev_order_date")))

df_time.select(col("Customer Name"),col("Order Date"),col("prev_order_date"),col("next_order_date"),col("Days_between_orders")).show()

In [0]:
data = [("West","Alice"),("East","Bob"),("South","Charlie"),("Central","David")]
manager_df = spark.createDataFrame(data,['Region',"Manager_name"])

display(manager_df)

In [0]:
# inner:
df_inner = df_unique.join(manager_df,on='Region',how='inner')
df_left = df_unique.join(manager_df,on='Region',how='left')
df_right = df_unique.join(manager_df,on='Region',how='right')
df_outer = df_unique.join(manager_df,on='Region',how='outer')
display(df_inner)

In [0]:
display(df_left)

In [0]:
df_missing_manager = df_unique.join(manager_df,on='Region',how='anti')
display(df_missing_manager)

In [0]:
from pyspark.sql.functions import split,explode,col

df_words = df_unique.withColumn("word_array",split(col("Product Name")," "))
df_words = df_words.withColumn("single_word",explode(col("word_array")))
display(df_words)

In [0]:
#Business Question: What are the total sales per Region, but I want Regions as COLUMNS?
pivot_df = df_unique.groupBy("Category").pivot("Region").sum("Sales")
display(pivot_df)

In [0]:
# create two halves 
west_df = df_unique.filter(col("Region")=="West")
east_df = df_unique.filter(col("Region")=="East")
# glue them back together
df_west_east = west_df.unionByName(east_df)
print(f"CombinedCount: {df_west_east.count()}")


In [0]:
df_unique.write.parquet("/Volumes/pyspark_pracitce/default/pyspark/superstore_parquet")


In [0]:
df_parquet = spark.read.parquet("/Volumes/pyspark_pracitce/default/pyspark/superstore_parquet")
display(df_parquet)

In [0]:
# Rename columns to remove spaces (Delta doesn't allow spaces in column names)
df_delta = df_unique
for col_name in df_unique.columns:
    df_delta = df_delta.withColumnRenamed(col_name, col_name.replace(" ", "_"))

df_delta.write.format("delta").saveAsTable("gold_superstore")

In [0]:
display(spark.sql("DESCRIBE HISTORY gold_superstore"))